<a href="https://colab.research.google.com/github/royalsflush/hackernews_vs_googletrends/blob/main/Hacker_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.cloud import bigquery
import pandas as pd

In [2]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

project_id = 'data-mining-project-505611'
client = bigquery.Client(project=project_id)

Authenticated


In [3]:
# Create story view
query = """
CREATE OR REPLACE VIEW hackernews_royalsflush.hackernews_story AS (
  SELECT title,
        url,
        text,
        `by`,
        score,
        `timestamp`,
        id,
        descendants
  FROM `bigquery-public-data.hacker_news.full`
  WHERE type = 'story'
  AND `by` IS NOT NULL
  AND dead IS NULL
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=4d24bc03-7e50-46e5-9253-d9b477ce5ad7>


In [4]:
# Create comments view
query = """
CREATE OR REPLACE VIEW hackernews_royalsflush.hackernews_comment AS (
  SELECT text,
       `by`,
       `timestamp`,
       id,
       parent,
  FROM `bigquery-public-data.hacker_news.full`
  WHERE type = 'comment'
  AND `by` IS NOT NULL
  AND dead IS NULL
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=79767495-4695-4150-9ecf-a7e9c7e79df2>


In [5]:
# Create story <-> comment ID mapping
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_comment_story_mapping AS (
  WITH RECURSIVE
  comment_id_pairs AS (
    SELECT id, parent
    FROM `data-mining-project-505611.hackernews_royalsflush.hackernews_comment`
  ),
  story_ids AS (
    SELECT id
    FROM `data-mining-project-505611.hackernews_royalsflush.hackernews_story`
  ),
  R AS (
    (SELECT id, parent AS ancestor from comment_id_pairs)
    UNION ALL (
      SELECT R.id, comment_id_pairs.parent AS ancestor
      FROM R
      INNER JOIN comment_id_pairs ON ancestor = comment_id_pairs.id
    )
  )
  SELECT R.id, ancestor
  FROM R
  INNER JOIN story_ids
  ON story_ids.id = R.ancestor
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=a27ad46c-e5bf-4559-a166-49e495f1b3d9>


In [9]:
!pip install requests

In [28]:
import concurrent.futures
import requests
import threading

query = """
SELECT id, url
FROM `hackernews_royalsflush.hackernews_story`
WHERE url IS NOT NULL
LIMIT 10;
"""

df = client.query(query).to_dataframe()
df.set_index('id', inplace=True)
df_lock = threading.Lock()
print(df)

headers = {
    'User-Agent': 'My User Agent 1.0',
    'From': 'youremail@domain.example'  # This is another valid field
}

# Modified example from ThreadPoolExecutor docs
def load_url(url, timeout):
    return requests.get(url, timeout=timeout, headers=headers)

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    # Start the load operations and mark each future with its URL
    future_to_url_id = {executor.submit(load_url, row['url'], 60): id for id, row in df.iterrows()}
    for future in concurrent.futures.as_completed(future_to_url_id):
        id  = future_to_url_id[future]
        #print(id)

        try:
            data = future.result()
        except Exception as exc:
            #print('%r generated an exception: %s' % (url, exc))
            with df_lock:
                df.loc[id, 'content'] = ""
                df.loc[id, 'url_error'] = True
        else:
            with df_lock:
                df.loc[id, 'content'] = data.text
                df.loc[id, 'url_error'] = False

print(df)

                                                       url  content
id                                                                 
9128522  https://en.wikipedia.org/wiki/Keynesian_beauty...        0
9128553                             https://www.roadie.com        1
9128580                 https://github.com/yahoo/dispatchr        2
9128603  http://www.engadget.com/2015/03/01/samsungs-pa...        3
9128611                              http://remembrall.me/        4
9128661                http://andrew.triumf.ca/FDE-VM.html        5
9128682  https://medium.com/@ayasin/reusing-your-restfu...        6
9128691  https://medium.com/on-startups/we-learn-from-e...        7
9128697  http://all3dp.com/print-customizable-movie-awa...        8
91287    http://blog.thembid.com/index.php/2007/12/19/d...        9


/tmp/ipykernel_1833/3582223658.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[id, 'content'] = ""


                                                       url  \
id                                                           
9128522  https://en.wikipedia.org/wiki/Keynesian_beauty...   
9128553                             https://www.roadie.com   
9128580                 https://github.com/yahoo/dispatchr   
9128603  http://www.engadget.com/2015/03/01/samsungs-pa...   
9128611                              http://remembrall.me/   
9128661                http://andrew.triumf.ca/FDE-VM.html   
9128682  https://medium.com/@ayasin/reusing-your-restfu...   
9128691  https://medium.com/on-startups/we-learn-from-e...   
9128697  http://all3dp.com/print-customizable-movie-awa...   
91287    http://blog.thembid.com/index.php/2007/12/19/d...   

                                                   content url_error  
id                                                                    
9128522  <!DOCTYPE html>\n<html class="client-nojs vect...     False  
9128553  <!doctype html><html lang="en"><h

In [ ]:
!pip install pytrends-modern

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.8 MB/s eta 0:00:00


In [ ]:
from pytrends_modern import TrendReq

pytrends = TrendReq(hl='en-US', tz=360)
pytrends.build_payload(
    kw_list=['Python', 'JavaScript'],
    timeframe='today 12-m',
    geo='US',
)

interest_df = pytrends.interest_over_time()
region_df = pytrends.interest_by_region()
related = pytrends.related_queries()

print(interest_df)

ResponseError: Request failed: HTTPSConnectionPool(host='trends.google.com', port=443): Max retries exceeded with url: /trends/api/widgetdata/relatedsearches?req=%7B%22restriction%22%3A+%7B%22geo%22%3A+%7B%22country%22%3A+%22US%22%7D%2C+%22time%22%3A+%222025-08-17+2026-08-17%22%2C+%22originalTimeRangeForExploreUrl%22%3A+%22today+12-m%22%2C+%22complexKeywordsRestriction%22%3A+%7B%22keyword%22%3A+%5B%7B%22type%22%3A+%22BROAD%22%2C+%22value%22%3A+%22JavaScript%22%7D%5D%7D%7D%2C+%22keywordType%22%3A+%22QUERY%22%2C+%22metric%22%3A+%5B%22TOP%22%2C+%22RISING%22%5D%2C+%22trendinessSettings%22%3A+%7B%22compareTime%22%3A+%222024-08-16+2025-08-16%22%7D%2C+%22requestOptions%22%3A+%7B%22property%22%3A+%22%22%2C+%22backend%22%3A+%22IZG%22%2C+%22category%22%3A+0%7D%2C+%22language%22%3A+%22en%22%2C+%22userCountryCode%22%3A+%22US%22%2C+%22userConfig%22%3A+%7B%22userType%22%3A+%22USER_TYPE_SCRAPER%22%7D%7D&token=ANI_2wMAAAAAaoRjQG_-k5x2-GPTchuZIvrwxuwKCgUf&tz=360 (Caused by ResponseError('too many 429 e